# E1 · The neighbors feature that contains the label of the point itself

**Type**: Find the bug · **Level**: ●●● challenge · **Time**: about 45 minutes · **extended path**

## Objective

Find out why a random forest to which "the label of the nearest neighbor" was added as a feature relies blindly on that column in training and does worse in test: for the **training** points the nearest neighbor is the point itself.
Fix it by excluding itself and measure how much the defect weighs.

## Dataset

`make_moons(1500, noise=0.4)` split 50/50: `X_tr, X_te, y_tr, y_te`. `nn = NearestNeighbors(n_neighbors=1).fit(X_tr)` searches the neighbors in the training set.

## Educational purpose

Recognize a typical bug of "feature engineering with neighbors" (KNN as a feature for stacking, local target means, spatial target encoding): any feature computed **from the target of the neighbors** must exclude the point itself,
otherwise it contains the answer.

## Task

1. Run the colleague's cell: `bug_test_acc` is lower than that of a random forest without the new feature.
2. Save `self_share` (fraction of training points whose 'nearest neighbor' returned by `nn.kneighbors(X_tr)` is itself) and `train_feature_acc` / `test_feature_acc` (accuracy of the feature alone used as a prediction, on the training and on the test set).
3. Write `neighbor_feature(nn, Xq, y_ref, exclude_self)` that returns the label of the nearest neighbor, skipping the point itself when `exclude_self` is true (you need to query `n_neighbors=2`, that is a new `NearestNeighbors`).
4. Rebuild the training feature with `exclude_self=True` and the test one with `False` (the test set is not in the training set): save `ok_test_acc`, and the importances of the new column `bug_imp` and `ok_imp`. Also save `X_only_acc` (random forest on the two coordinates only).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

X, y = make_moons(1500, noise=0.4, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0)
nn = NearestNeighbors(n_neighbors=1).fit(X_tr)
rf = lambda: RandomForestClassifier(200, random_state=0, n_jobs=-1)

## Your work

### The colleague's code

In [ ]:
f_tr = y_tr[nn.kneighbors(X_tr)[1][:, 0]]            # label of the nearest neighbor, for every training point
f_te = y_tr[nn.kneighbors(X_te)[1][:, 0]]            # and for every test point
A_tr, A_te = np.c_[X_tr, f_tr], np.c_[X_te, f_te]
bug_model = rf().fit(A_tr, y_tr)
bug_test_acc = bug_model.score(A_te, y_te)
bug_imp = bug_model.feature_importances_[-1]
print(f"test accuracy: {bug_test_acc:.3f} | importance of the new column: {bug_imp:.2f}")

### Your diagnosis

In [ ]:
self_share = None
train_feature_acc = None
test_feature_acc = None

In [ ]:
def neighbor_feature(nn, Xq, y_ref, exclude_self):
    ...

In [ ]:
ok_test_acc = None
ok_imp = None
X_only_acc = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`nn2 = NearestNeighbors(n_neighbors=2).fit(X_tr); idx = nn2.kneighbors(X_tr)[1][:, 1]` (column 1: the first neighbor other than itself, since itself is at distance 0). For the test set there is nothing to skip: no test point is in the training set.'):
    check(bug_test_acc < 0.82, "the colleague's model is below 0.82 on the test set", "don't modify the colleague's code")
    check(self_share == 1.0 and train_feature_acc == 1.0, "for all the training points the 'nearest neighbor' is itself: the feature is the label", "compare the returned indices with np.arange(len(X_tr))")
    check(test_feature_acc < 0.85, "on the test set the same feature is much less accurate (below 0.85)", "the feature only holds for the label of the neighbor in the training set")
    check(np.array_equal(neighbor_feature(nn, X_tr, y_tr, True), y_tr[NearestNeighbors(n_neighbors=2).fit(X_tr).kneighbors(X_tr)[1][:, 1]]) and np.array_equal(neighbor_feature(nn, X_te, y_tr, False), f_te), "neighbor_feature skips itself only when needed", "for the training set query two neighbors and take the second")
    check(ok_test_acc > bug_test_acc + 0.02, "with the correct feature the test accuracy goes up by over 0.02", "exclude_self=True for the training set, False for the test set")
    check(bug_imp > 2.5 * ok_imp, "the colleague's model gives the new column over 2.5 times the importance of the correct one", "feature_importances_[-1]")
    check(abs(ok_test_acc - X_only_acc) < 0.03, "the neighbors feature, done right, adds nothing to the coordinates (gap below 0.03)", "the forest already had the two coordinates")
done()

---
The commented solution is in `solutions/E_find_the_bug/E1_the_nearest_neighbor_is_you.ipynb`. Open it after you pass the check, or after you have really tried.